In [1]:
import pandas as pd

amazon_df = pd.read_csv(
    "../data/processed/amazonhelp_cleaned.csv"
)

print("Shape:", amazon_df.shape)

number_of_conversations = amazon_df["conversation_id"].nunique()

print("Number of conversations:", number_of_conversations)

conversation_lengths = (
    amazon_df
    .groupby("conversation_id")
    .size()
    .rename("conversation_length")
)

print(conversation_lengths.describe())

length_distribution = conversation_lengths.value_counts().sort_index()

print(length_distribution)

length_groups = pd.cut(
    conversation_lengths,
    bins=[0, 1, 2, 3, 4, float("inf")],
    labels=["1 turn", "2 turns", "3 turns", "4 turns", "5+ turns"]
)

length_summary = length_groups.value_counts().sort_index()

print(length_summary)

length_percentage = (
    length_summary
    / length_summary.sum()
    * 100
)

length_distribution_table = pd.DataFrame({
    "conversations": length_summary,
    "percentage": length_percentage.round(2)
})

print(length_distribution_table)

amazonhelp_conversations = (
    amazon_df[
        amazon_df["author_id"] == "AmazonHelp"
    ]["conversation_id"]
    .nunique()
)

print(
    "Conversations containing AmazonHelp responses:",
    amazonhelp_conversations
)

amazonhelp_percentage = (
    amazonhelp_conversations
    / number_of_conversations
    * 100
)

print(
    "Percentage:",
    round(amazonhelp_percentage, 2),
    "%"
)

message_roles = amazon_df["inbound"].value_counts()

print(message_roles)

customer_messages = (
    amazon_df["inbound"] == True
).sum()

amazonhelp_messages = (
    amazon_df["inbound"] == False
).sum()

print("Customer messages:", customer_messages)
print("AmazonHelp responses:", amazonhelp_messages)

conversation_role_summary = (
    amazon_df
    .groupby("conversation_id")
    .agg(
        total_messages=("tweet_id", "count"),
        customer_messages=("inbound", "sum")
    )
)

conversation_role_summary["amazonhelp_responses"] = (
    conversation_role_summary["total_messages"]
    - conversation_role_summary["customer_messages"]
)

conversation_role_summary.head()

two_sided_conversations = conversation_role_summary[
    (conversation_role_summary["customer_messages"] > 0) &
    (conversation_role_summary["amazonhelp_responses"] > 0)
]

print(
    "Two-sided conversations:",
    len(two_sided_conversations)
)

two_sided_percentage = (
    len(two_sided_conversations)
    / number_of_conversations
    * 100
)

print(
    "Percentage of two-sided conversations:",
    round(two_sided_percentage, 2),
    "%"
)

Shape: (374026, 10)
Number of conversations: 82534
count    82534.000000
mean         4.531781
std          5.321938
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max        448.000000
Name: conversation_length, dtype: float64
conversation_length
1          1
2      31292
3      11570
4      14014
5       6063
       ...  
161        1
207        1
218        1
425        1
448        2
Name: count, Length: 99, dtype: int64
conversation_length
1 turn          1
2 turns     31292
3 turns     11570
4 turns     14014
5+ turns    25657
Name: count, dtype: int64
                     conversations  percentage
conversation_length                           
1 turn                           1        0.00
2 turns                      31292       37.91
3 turns                      11570       14.02
4 turns                      14014       16.98
5+ turns                     25657       31.09
Conversations containing AmazonHelp responses: 82534
Percentage: 

In [2]:
import re

intent_keywords = {

    # --------------------------------------------------------
    # 4.1 Delivery
    # --------------------------------------------------------
    "DELIVERY_DELAY": [
        "late delivery",
        "delivery delayed",
        "delivery delay",
        "delayed delivery",
        "arriving late",
        "arrived late"
    ],

    "DELIVERY_NOT_RECEIVED": [
        "not received",
        "didn't receive",
        "did not receive",
        "haven't received",
        "have not received",
        "never received",
        "package hasn't arrived",
        "package has not arrived"
    ],

    "DELIVERED_BUT_MISSING": [
        "marked delivered",
        "shows delivered",
        "says delivered",
        "delivered but",
        "delivered and missing",
        "says it was delivered"
    ],

    "DELIVERY_LOCATION_PROBLEM": [
        "wrong address",
        "wrong location",
        "wrong place",
        "delivered to neighbor",
        "delivered to neighbour",
        "delivery location",
        "delivery instructions"
    ],

    "COURIER_OR_DRIVER_ISSUE": [
        "delivery driver",
        "driver",
        "courier",
        "delivery person",
        "delivery guy",
        "driver was",
        "courier was"
    ],

    "DELIVERY_TRACKING_PROBLEM": [
        "tracking",
        "tracking number",
        "tracking hasn't updated",
        "tracking has not updated",
        "tracking not updating",
        "tracking information"
    ],

    "DELIVERY_ATTEMPT_PROBLEM": [
        "delivery attempt",
        "attempted delivery",
        "delivery attempted",
        "couldn't deliver",
        "could not deliver",
        "delivery failed"
    ],

    "DAMAGED_DURING_DELIVERY": [
        "damaged during delivery",
        "damaged package",
        "package damaged",
        "arrived damaged",
        "box damaged",
        "package was damaged"
    ],


    # --------------------------------------------------------
    # 4.2 Orders
    # --------------------------------------------------------
    "ORDER_CANCELLATION": [
        "cancel my order",
        "cancel order",
        "order cancellation",
        "cancelled my order",
        "canceled my order"
    ],

    "ORDER_PLACEMENT_PROBLEM": [
        "can't place order",
        "cannot place order",
        "unable to place order",
        "can't order",
        "cannot order",
        "place my order",
        "place an order"
    ],

    "ORDER_STATUS": [
        "order status",
        "where is my order",
        "order update",
        "order hasn't arrived",
        "order has not arrived",
        "check my order"
    ],

    "ORDER_MODIFICATION": [
        "change my order",
        "modify my order",
        "change order",
        "modify order",
        "edit my order"
    ],

    "WRONG_OR_MISSING_ITEM": [
        "wrong item",
        "wrong product",
        "missing item",
        "item missing",
        "missing product",
        "received the wrong"
    ],

    "PRODUCT_AVAILABILITY": [
        "out of stock",
        "out-of-stock",
        "not available",
        "product availability",
        "when will it be available",
        "back in stock"
    ],


    # --------------------------------------------------------
    # 4.3 Returns & Refunds
    # --------------------------------------------------------
    "RETURN_REQUEST": [
        "return this",
        "return my",
        "return an item",
        "want to return",
        "want a return",
        "return request"
    ],

    "RETURN_ELIGIBILITY": [
        "eligible for return",
        "return eligible",
        "can I return",
        "can i return",
        "return eligibility",
        "eligible to return"
    ],

    "RETURN_PICKUP_PROBLEM": [
        "return pickup",
        "return pick up",
        "pickup didn't happen",
        "pickup did not happen",
        "pickup hasn't happened",
        "courier didn't pick up",
        "courier did not pick up"
    ],

    "RETURN_PROBLEM": [
        "return problem",
        "return issue",
        "problem with my return",
        "issue with my return",
        "return isn't working",
        "return is not working"
    ],

    "REFUND_PROBLEM": [
        "refund",
        "refund problem",
        "refund issue",
        "refund hasn't arrived",
        "refund has not arrived",
        "refund not received",
        "where is my refund"
    ],

    "REPLACEMENT_PROBLEM": [
        "replacement",
        "replacement problem",
        "replacement issue",
        "replacement hasn't arrived",
        "replacement has not arrived",
        "replace my item"
    ],


    # --------------------------------------------------------
    # 4.4 Payment & Billing
    # --------------------------------------------------------
    "PAYMENT_FAILURE": [
        "payment failed",
        "payment failure",
        "payment declined",
        "payment was declined",
        "can't make payment",
        "cannot make payment"
    ],

    "CHARGE_PROBLEM": [
        "charged",
        "charge problem",
        "wrong charge",
        "unexpected charge",
        "incorrect charge",
        "charged me"
    ],

    "DUPLICATE_CHARGE": [
        "charged twice",
        "charged two times",
        "double charge",
        "duplicate charge",
        "charged me twice"
    ],

    "UNAUTHORIZED_PAYMENT": [
        "unauthorized payment",
        "unauthorized charge",
        "payment I didn't make",
        "payment i did not make",
        "charge I didn't make",
        "charge i did not make"
    ],

    "AMAZON_PAY_PROBLEM": [
        "amazon pay",
        "amazonpay",
        "amazon pay problem",
        "amazon pay issue"
    ],


    # --------------------------------------------------------
    # 4.5 Prime & Subscriptions
    # --------------------------------------------------------
    "PRIME_MEMBERSHIP_PROBLEM": [
        "prime membership",
        "prime membership problem",
        "prime membership issue",
        "prime member"
    ],

    "PRIME_CHARGE_OR_RENEWAL": [
        "prime charge",
        "prime charged",
        "prime renewal",
        "prime renewed",
        "prime renewal charge",
        "charged for prime"
    ],

    "PRIME_BENEFIT_PROBLEM": [
        "prime benefit",
        "prime benefits",
        "prime benefit not working",
        "prime benefits not working",
        "prime isn't working",
        "prime is not working"
    ],

    "SUBSCRIPTION_PROBLEM": [
        "subscription",
        "subscription problem",
        "subscription issue",
        "cancel subscription",
        "subscription charge",
        "subscription renewed"
    ],


    # --------------------------------------------------------
    # 4.6 Technical / Digital
    # --------------------------------------------------------
    "DEVICE_PROBLEM": [
        "echo",
        "alexa",
        "kindle",
        "fire tv",
        "device problem",
        "device issue",
        "device isn't working",
        "device is not working"
    ],

    "APP_OR_WEBSITE_PROBLEM": [
        "amazon app",
        "app problem",
        "app issue",
        "website problem",
        "website issue",
        "website isn't working",
        "website is not working"
    ],

    "DIGITAL_CONTENT_PROBLEM": [
        "digital content",
        "ebook",
        "e-book",
        "kindle book",
        "digital book",
        "digital purchase"
    ],

    "STREAMING_OR_PLAYBACK_PROBLEM": [
        "streaming",
        "playback",
        "video not playing",
        "music not playing",
        "video playback",
        "can't watch",
        "cannot watch"
    ],

    "ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM": [
        "connect",
        "connection problem",
        "connectivity",
        "connected",
        "link my account",
        "account linking",
        "linking account"
    ],


    # --------------------------------------------------------
    # 4.7 Security & Fraud
    # --------------------------------------------------------
    "SUSPICIOUS_EMAIL_OR_PHISHING": [
        "phishing",
        "suspicious email",
        "suspicious e-mail",
        "fake email",
        "fake e-mail",
        "is this email from amazon",
        "is this a real email"
    ],

    "SCAM_OR_FRAUD_CONCERN": [
        "scam",
        "fraud",
        "fraudulent",
        "scammed",
        "scammer",
        "fraud concern"
    ],

    "UNAUTHORIZED_ACCOUNT_ACTIVITY": [
        "unauthorized account",
        "someone accessed my account",
        "account hacked",
        "account was hacked",
        "someone logged into my account",
        "someone has access to my account"
    ],


    # --------------------------------------------------------
    # 4.8 Promotions & Offers
    # --------------------------------------------------------
    "PROMOTION_OR_DISCOUNT_PROBLEM": [
        "promotion",
        "promotional",
        "discount",
        "promo code",
        "promo",
        "discount code"
    ],

    "CASHBACK_PROBLEM": [
        "cashback",
        "cash back",
        "cashback missing",
        "cashback not received"
    ],

    "CONTEST_OR_QUIZ_INQUIRY": [
        "contest",
        "quiz",
        "contest winner",
        "quiz winner",
        "contest result",
        "quiz result"
    ],

    "OFFER_ELIGIBILITY": [
        "offer eligibility",
        "eligible for this offer",
        "eligible for the offer",
        "qualify for the offer",
        "qualify for this promotion"
    ],


    # --------------------------------------------------------
    # 4.9 Customer Service
    # --------------------------------------------------------
    "CUSTOMER_SERVICE_COMPLAINT": [
        "customer service",
        "customer support",
        "support was",
        "support team",
        "bad customer service",
        "poor customer service"
    ],

    "SUPPORT_AGENT_COMPLAINT": [
        "agent",
        "support agent",
        "customer service agent",
        "agent was rude",
        "rude agent",
        "agent didn't help",
        "agent did not help"
    ],

    "UNRESOLVED_SUPPORT_ISSUE": [
        "not resolved",
        "still not resolved",
        "issue isn't resolved",
        "issue is not resolved",
        "problem still exists",
        "still having this problem",
        "no solution"
    ],

    "ESCALATION_REQUEST": [
        "escalate",
        "escalation",
        "escalate this",
        "speak to a manager",
        "talk to a manager",
        "supervisor"
    ],


    # --------------------------------------------------------
    # 4.10 General / Other
    # --------------------------------------------------------
    "GENERAL_INFORMATION": [
        "information",
        "question",
        "how does",
        "how do I",
        "how can I"
    ],

    "PRODUCT_INFORMATION": [
        "product information",
        "product details",
        "product question",
        "tell me about this product",
        "does this product"
    ],

    "FEEDBACK": [
        "feedback",
        "suggestion",
        "suggest",
        "recommendation"
    ],

    "APPRECIATION": [
        "thank you amazon",
        "thanks amazon",
        "thank you",
        "thanks",
        "great service",
        "excellent service"
    ],

    "UNCLEAR_REQUEST": []
}


# ============================================================
# Find candidate conversations for each intent
# ============================================================

family_candidates = {}

for intent, keywords in intent_keywords.items():

    # If no keywords are defined
    if not keywords:
        family_candidates[intent] = pd.Series(
            dtype="object"
        )
        continue

    # Escape keywords so special regex characters
    # do not cause problems
    escaped_keywords = [
        re.escape(keyword)
        for keyword in keywords
    ]

    pattern = "|".join(escaped_keywords)

    mask = (
        amazon_df["clean_text"]
        .fillna("")
        .str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    )

    family_candidates[intent] = (
        amazon_df.loc[mask, "conversation_id"]
        .dropna()
        .drop_duplicates()
    )


# ============================================================
# Display candidate counts
# ============================================================

sampling_summary = pd.DataFrame({
    "intent": list(family_candidates.keys()),
    "candidate_conversations": [
        len(family_candidates[intent])
        for intent in family_candidates
    ]
})

sampling_summary = sampling_summary.sort_values(
    "candidate_conversations",
    ascending=False
)

print(sampling_summary.to_string(index=False))

                                 intent  candidate_conversations
                           APPRECIATION                    16111
                    GENERAL_INFORMATION                    13867
             CUSTOMER_SERVICE_COMPLAINT                    10458
                               FEEDBACK                     6440
                         REFUND_PROBLEM                     5842
              DELIVERY_TRACKING_PROBLEM                     4822
                         DEVICE_PROBLEM                     4686
                COURIER_OR_DRIVER_ISSUE                     4571
                  DELIVERY_NOT_RECEIVED                     4047
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM                     3121
               PRIME_MEMBERSHIP_PROBLEM                     2523
                    REPLACEMENT_PROBLEM                     1983
                     ESCALATION_REQUEST                     1976
          PROMOTION_OR_DISCOUNT_PROBLEM                     1405
                         

In [3]:
candidate_rows = []

for intent, conversation_ids in family_candidates.items():

    # Skip intents with no candidate conversations
    if len(conversation_ids) == 0:
        continue

    for conversation_id in conversation_ids:

        # Get complete conversation
        conversation = amazon_df[
            amazon_df["conversation_id"] == conversation_id
        ].sort_values("created_at")

        # Build readable conversation text
        conversation_text = "\n".join(
            (
                "Customer: " if row["inbound"]
                else "AmazonHelp: "
            ) + str(row["clean_text"])
            for _, row in conversation.iterrows()
        )

        # Conversation length
        conversation_length = len(conversation)

        candidate_rows.append({
            "conversation_id": conversation_id,
            "conversation_length": conversation_length,
            "candidate_intent": intent,
            "candidate_source": "keyword_rule",
            "conversation_text": conversation_text
        })


# Create candidate DataFrame
candidate_pool = pd.DataFrame(candidate_rows)


# Remove duplicate conversation + candidate intent combinations
candidate_pool = candidate_pool.drop_duplicates(
    subset=[
        "conversation_id",
        "candidate_intent"
    ]
)


print("Candidate pool shape:", candidate_pool.shape)

print(
    "Unique conversations:",
    candidate_pool["conversation_id"].nunique()
)

print(
    "Unique candidate intents:",
    candidate_pool["candidate_intent"].nunique()
)

Candidate pool shape: (97553, 5)
Unique conversations: 50108
Unique candidate intents: 49


In [4]:
# ============================================================
# Cell 12 — Inspect Candidate Pool
# ============================================================

candidate_pool[
    [
        "conversation_id",
        "conversation_length",
        "candidate_intent",
        "candidate_source",
        "conversation_text"
    ]
].head(10)

,conversation_id,conversation_length,candidate_intent,candidate_source,conversation_text
0,2559,6,DELIVERY_DELAY,keyword_rule,"Customer: @AmazonHelp hello, I’m having an iss..."
1,11073,6,DELIVERY_DELAY,keyword_rule,Customer: @AmazonHelp isn't prime two day ship...
2,19137,218,DELIVERY_DELAY,keyword_rule,Customer: Don't get spooked by long delivery t...
3,42587,6,DELIVERY_DELAY,keyword_rule,Customer: @115821 I normally love you but the ...
4,77335,8,DELIVERY_DELAY,keyword_rule,Customer: Another delayed delivery @115830 ? E...
5,78873,6,DELIVERY_DELAY,keyword_rule,Customer: Items ordered on #CyberMonday from @...
6,86108,12,DELIVERY_DELAY,keyword_rule,Customer: @AmazonHelp Prime’s 2 day shipping a...
7,86208,4,DELIVERY_DELAY,keyword_rule,Customer: Today is the twins’ 4th Birthday. Th...
8,89101,2,DELIVERY_DELAY,keyword_rule,Customer: The whole perk of @115821 Prime is 2...
9,89316,4,DELIVERY_DELAY,keyword_rule,Customer: If I hear “it’s not our fault your p...


In [5]:

unique_candidate_ids = (
    candidate_pool["conversation_id"]
    .drop_duplicates()
)

candidate_sample_ids = unique_candidate_ids.sample(
    n=min(500, len(unique_candidate_ids)),
    random_state=42
)

candidate_pool_500 = candidate_pool[
    candidate_pool["conversation_id"].isin(
        candidate_sample_ids
    )
].copy()

print(
    "Unique candidate conversations:",
    candidate_pool_500["conversation_id"].nunique()
)

print(
    "Total candidate rows:",
    len(candidate_pool_500)
)

Unique candidate conversations: 500
Total candidate rows: 976


In [6]:
# ============================================================
# Cell 15 — Candidate Distribution by Intent
# ============================================================

candidate_distribution = (
    candidate_pool
    .groupby("candidate_intent")
    .size()
    .reset_index(name="candidate_count")
    .sort_values(
        "candidate_count",
        ascending=False
    )
)

print(candidate_distribution.to_string(index=False))

                       candidate_intent  candidate_count
                           APPRECIATION            16111
                    GENERAL_INFORMATION            13867
             CUSTOMER_SERVICE_COMPLAINT            10458
                               FEEDBACK             6440
                         REFUND_PROBLEM             5842
              DELIVERY_TRACKING_PROBLEM             4822
                         DEVICE_PROBLEM             4686
                COURIER_OR_DRIVER_ISSUE             4571
                  DELIVERY_NOT_RECEIVED             4047
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM             3121
               PRIME_MEMBERSHIP_PROBLEM             2523
                    REPLACEMENT_PROBLEM             1983
                     ESCALATION_REQUEST             1976
          PROMOTION_OR_DISCOUNT_PROBLEM             1405
                         CHARGE_PROBLEM             1383
                SUPPORT_AGENT_COMPLAINT             1318
                  SCAM_OR_FRAUD

In [7]:
# ============================================================
# Cell 16 — Candidate Distribution with Percentage
# ============================================================

total_candidate_rows = candidate_distribution["candidate_count"].sum()

candidate_distribution["percentage"] = (
    candidate_distribution["candidate_count"]
    / total_candidate_rows
    * 100
).round(2)

print(
    candidate_distribution.to_string(index=False)
)

                       candidate_intent  candidate_count  percentage
                           APPRECIATION            16111       16.52
                    GENERAL_INFORMATION            13867       14.21
             CUSTOMER_SERVICE_COMPLAINT            10458       10.72
                               FEEDBACK             6440        6.60
                         REFUND_PROBLEM             5842        5.99
              DELIVERY_TRACKING_PROBLEM             4822        4.94
                         DEVICE_PROBLEM             4686        4.80
                COURIER_OR_DRIVER_ISSUE             4571        4.69
                  DELIVERY_NOT_RECEIVED             4047        4.15
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM             3121        3.20
               PRIME_MEMBERSHIP_PROBLEM             2523        2.59
                    REPLACEMENT_PROBLEM             1983        2.03
                     ESCALATION_REQUEST             1976        2.03
          PROMOTION_OR_DISCOUNT_PR

In [8]:
for conversation_id, group in candidate_pool_500.groupby(
    "conversation_id"
):

    print("=" * 100)
    print(f"Conversation: {conversation_id}")
    print("=" * 100)

    print(
        "Conversation length:",
        group["conversation_length"].iloc[0]
    )

    print(
        "Candidate intent(s):",
        ", ".join(group["candidate_intent"].unique())
    )

    print("\nConversation:")
    print(group["conversation_text"].iloc[0])

    print("\n")

Conversation: 16758
Conversation length: 6
Candidate intent(s): REFUND_PROBLEM

Conversation:
Customer: @AmazonHelp fuck you guys. Canceling Prime. Not gonna do business with you guys ever again.
AmazonHelp: @119697 This isn't how we want you to feel. We'd like to help, if possible. Please let us know more about what's going.
Customer: @AmazonHelp No one knows what to do to get my refund back to me!
Customer: @AmazonHelp You know it's some fucked up shit when you gotta conference call @116827 and @AmazonHelp
AmazonHelp: @119697 I'm sorry for the trouble. Generally, the bank can issue a refund to the new active card on file. Please keep us updated.
Customer: @AmazonHelp No card on file after 30 days. Refund send back to merchant. That's you.


Conversation: 21586
Conversation length: 53
Candidate intent(s): DELIVERY_NOT_RECEIVED, COURIER_OR_DRIVER_ISSUE, PRODUCT_AVAILABILITY, REFUND_PROBLEM, CUSTOMER_SERVICE_COMPLAINT, SUPPORT_AGENT_COMPLAINT, ESCALATION_REQUEST, GENERAL_INFORMATION, FE

In [9]:
golden_set = pd.DataFrame(
    columns=[
        "conversation_id",
        "conversation_text",
        "candidate_intent",
        "gold_intent",
        "label_reason",
        "difficulty"
    ]
)

In [10]:
sampling_plan = candidate_distribution.copy()

# Classify each intent
def classify_group(count):
    if count > 1000:
        return "Common"
    elif count >= 100:
        return "Medium"
    elif count > 0:
        return "Rare"
    else:
        return "No candidates"


sampling_plan["group"] = (
    sampling_plan["candidate_count"]
    .apply(classify_group)
)


# Initial allocation
def initial_target(group):
    if group == "Common":
        return 6
    elif group == "Medium":
        return 4
    elif group == "Rare":
        return 2
    else:
        return 0


sampling_plan["target_samples"] = (
    sampling_plan["group"]
    .apply(initial_target)
)


print(sampling_plan.to_string(index=False))

print(
    "\nInitial total:",
    sampling_plan["target_samples"].sum()
)

                       candidate_intent  candidate_count  percentage  group  target_samples
                           APPRECIATION            16111       16.52 Common               6
                    GENERAL_INFORMATION            13867       14.21 Common               6
             CUSTOMER_SERVICE_COMPLAINT            10458       10.72 Common               6
                               FEEDBACK             6440        6.60 Common               6
                         REFUND_PROBLEM             5842        5.99 Common               6
              DELIVERY_TRACKING_PROBLEM             4822        4.94 Common               6
                         DEVICE_PROBLEM             4686        4.80 Common               6
                COURIER_OR_DRIVER_ISSUE             4571        4.69 Common               6
                  DELIVERY_NOT_RECEIVED             4047        4.15 Common               6
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM             3121        3.20 Common     

In [11]:
# ============================================================
# Final Golden Set Sampling Plan
# ============================================================

# Start from the candidate distribution
sampling_plan = candidate_distribution.copy()


# ------------------------------------------------------------
# 1. Classify intents into Common / Medium / Rare
# ------------------------------------------------------------

def classify_group(count):

    if count > 1000:
        return "Common"

    elif count >= 100:
        return "Medium"

    elif count > 0:
        return "Rare"

    else:
        return "No candidates"


sampling_plan["group"] = (
    sampling_plan["candidate_count"]
    .apply(classify_group)
)


# ------------------------------------------------------------
# 2. Assign stratified target samples
# ------------------------------------------------------------

def assign_target(group):

    if group == "Common":
        return 5

    elif group == "Medium":
        return 4

    elif group == "Rare":
        return 2

    else:
        return 0


sampling_plan["target_samples"] = (
    sampling_plan["group"]
    .apply(assign_target)
)


# ------------------------------------------------------------
# 3. Mark these as STRATIFIED
# ------------------------------------------------------------

sampling_plan["sampling_type"] = "STRATIFIED"


# ------------------------------------------------------------
# 4. Add 7 difficult cases
# ------------------------------------------------------------

difficult_row = pd.DataFrame({
    "candidate_intent": ["DIFFICULT_CASES"],
    "candidate_count": [None],
    "percentage": [None],
    "group": ["—"],
    "target_samples": [7],
    "sampling_type": ["DIFFICULT"]
})


# Add difficult row
sampling_plan = pd.concat(
    [
        sampling_plan,
        difficult_row
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 5. Arrange columns
# ------------------------------------------------------------

sampling_plan = sampling_plan[
    [
        "candidate_intent",
        "candidate_count",
        "group",
        "target_samples",
        "sampling_type"
    ]
]


# ------------------------------------------------------------
# 6. Sort intents
# ------------------------------------------------------------

# Keep DIFFICULT_CASES at the bottom
normal_intents = sampling_plan[
    sampling_plan["candidate_intent"] != "DIFFICULT_CASES"
].sort_values(
    "candidate_count",
    ascending=False
)

difficult_cases = sampling_plan[
    sampling_plan["candidate_intent"] == "DIFFICULT_CASES"
]

sampling_plan = pd.concat(
    [
        normal_intents,
        difficult_cases
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Display final sampling plan
# ------------------------------------------------------------

print(
    sampling_plan.to_string(index=False)
)


# ------------------------------------------------------------
# 8. Verify total
# ------------------------------------------------------------

total_samples = sampling_plan["target_samples"].sum()

print("\nTotal target samples:", total_samples)

assert total_samples == 200, (
    f"Expected 200 samples, but got {total_samples}"
)

print("✅ Sampling plan is exactly 200.")

                       candidate_intent candidate_count  group  target_samples sampling_type
                           APPRECIATION           16111 Common               5    STRATIFIED
                    GENERAL_INFORMATION           13867 Common               5    STRATIFIED
             CUSTOMER_SERVICE_COMPLAINT           10458 Common               5    STRATIFIED
                               FEEDBACK            6440 Common               5    STRATIFIED
                         REFUND_PROBLEM            5842 Common               5    STRATIFIED
              DELIVERY_TRACKING_PROBLEM            4822 Common               5    STRATIFIED
                         DEVICE_PROBLEM            4686 Common               5    STRATIFIED
                COURIER_OR_DRIVER_ISSUE            4571 Common               5    STRATIFIED
                  DELIVERY_NOT_RECEIVED            4047 Common               5    STRATIFIED
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM            3121 Common        

C:\Users\Nihelesh M U\AppData\Local\Temp\ipykernel_12352\582712094.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sampling_plan = pd.concat(


In [12]:
# ============================================================
# Select Golden Set Conversations
# ============================================================

import pandas as pd


selected_samples = []


# ------------------------------------------------------------
# 1. Loop through every intent in the sampling plan
# ------------------------------------------------------------

for _, plan_row in sampling_plan.iterrows():

    intent = plan_row["candidate_intent"]
    target_samples = int(plan_row["target_samples"])

    # Skip the artificial DIFFICULT_CASES row for now
    if intent == "DIFFICULT_CASES":
        continue

    # Skip intents with zero target samples
    if target_samples == 0:
        continue

    # --------------------------------------------------------
    # Find conversations belonging to this candidate intent
    # --------------------------------------------------------

    candidates = candidate_pool[
        candidate_pool["candidate_intent"] == intent
    ].copy()

    # Remove duplicate conversation IDs
    candidates = candidates.drop_duplicates(
        subset=["conversation_id"]
    )

    # --------------------------------------------------------
    # Check that enough candidates exist
    # --------------------------------------------------------

    if len(candidates) < target_samples:

        print(
            f"WARNING: {intent} has only "
            f"{len(candidates)} candidates, "
            f"but {target_samples} were requested."
        )

        target_samples = len(candidates)

    # --------------------------------------------------------
    # Randomly select conversations
    # --------------------------------------------------------

    selected = candidates.sample(
        n=target_samples,
        random_state=42
    ).copy()

    # --------------------------------------------------------
    # Get information from sampling plan
    # --------------------------------------------------------

    selected["candidate_count"] = (
        plan_row["candidate_count"]
    )

    selected["group"] = (
        plan_row["group"]
    )

    selected["sampling_type"] = (
        "STRATIFIED"
    )

    # --------------------------------------------------------
    # Initially leave gold-label fields empty
    # --------------------------------------------------------

    selected["gold_intent"] = pd.NA
    selected["label_reason"] = pd.NA
    selected["difficulty"] = pd.NA

    # --------------------------------------------------------
    # Keep required columns
    # --------------------------------------------------------

    selected = selected[
        [
            "conversation_id",
            "candidate_intent",
            "candidate_count",
            "group",
            "sampling_type",
            "gold_intent",
            "label_reason",
            "difficulty"
        ]
    ]

    selected_samples.append(selected)


# ------------------------------------------------------------
# 2. Combine all sampled conversations
# ------------------------------------------------------------

golden_set = pd.concat(
    selected_samples,
    ignore_index=True
)


# ------------------------------------------------------------
# 3. Remove accidental duplicate conversations
# ------------------------------------------------------------

golden_set = golden_set.drop_duplicates(
    subset=["conversation_id"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. Display result
# ------------------------------------------------------------

print(
    "Selected conversations:",
    len(golden_set)
)

print(
    "Unique conversations:",
    golden_set["conversation_id"].nunique()
)

print("\nSampling type:")
print(
    golden_set["sampling_type"].value_counts()
)

print("\nCandidate intent distribution:")
print(
    golden_set["candidate_intent"].value_counts()
)

Selected conversations: 192
Unique conversations: 192

Sampling type:
sampling_type
STRATIFIED    192
Name: count, dtype: int64

Candidate intent distribution:
candidate_intent
APPRECIATION                               5
GENERAL_INFORMATION                        5
CUSTOMER_SERVICE_COMPLAINT                 5
FEEDBACK                                   5
REFUND_PROBLEM                             5
DELIVERY_TRACKING_PROBLEM                  5
DEVICE_PROBLEM                             5
COURIER_OR_DRIVER_ISSUE                    5
DELIVERY_NOT_RECEIVED                      5
ACCOUNT_LINKING_OR_CONNECTIVITY_PROBLEM    5
PRIME_MEMBERSHIP_PROBLEM                   5
REPLACEMENT_PROBLEM                        5
ESCALATION_REQUEST                         5
PROMOTION_OR_DISCOUNT_PROBLEM              5
CHARGE_PROBLEM                             5
SUPPORT_AGENT_COMPLAINT                    5
SCAM_OR_FRAUD_CONCERN                      5
PRODUCT_AVAILABILITY                       4
DELIVERED_BUT

In [13]:
# ============================================================
# Add conversation text
# ============================================================

conversation_texts = (
    candidate_pool[
        [
            "conversation_id",
            "conversation_text"
        ]
    ]
    .drop_duplicates("conversation_id")
)


golden_set = golden_set.merge(
    conversation_texts,
    on="conversation_id",
    how="left"
)

In [14]:
# ============================================================
# Final Golden Set column order
# ============================================================

golden_set = golden_set[
    [
        "conversation_id",
        "conversation_text",
        "candidate_intent",
        "candidate_count",
        "group",
        "sampling_type",
        "gold_intent",
        "label_reason",
        "difficulty"
    ]
]

print(golden_set.shape)

golden_set.head()

(192, 9)


,conversation_id,conversation_text,candidate_intent,candidate_count,group,sampling_type,gold_intent,label_reason,difficulty
0,2447398,Customer: @115821 thanks for the gift card tha...,APPRECIATION,16111,Common,STRATIFIED,NaN,NaN,NaN
1,2686757,Customer: the second prime order in a row that...,APPRECIATION,16111,Common,STRATIFIED,NaN,NaN,NaN
2,2967720,Customer: @115821 how can I apply for your NC ...,APPRECIATION,16111,Common,STRATIFIED,NaN,NaN,NaN
3,930181,Customer: @AmazonHelp Recently moved from Kind...,APPRECIATION,16111,Common,STRATIFIED,NaN,NaN,NaN
4,1132075,Customer: hi @AmazonHelp twitter rejected my r...,APPRECIATION,16111,Common,STRATIFIED,NaN,NaN,NaN


In [15]:
golden_set.to_csv(
    "../data/processed/amazonhelp_golden_set.csv",
    index=False
)

print("✅ Golden Set saved.")

PermissionError: [Errno 13] Permission denied: '../data/processed/amazonhelp_golden_set.csv'

In [16]:
# ============================================================
# Display complete conversations for the selected 200 IDs
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Selected 200 conversation IDs
# ------------------------------------------------------------

selected_conversation_ids = [
    2447398,
    2686757,
    2967720,
    930181,
    1132075,
    1551751,
    346732,
    1447950,
    863820,
    525543,
    1231224,
    2841686,
    2369649,
    1699857,
    2283867,
    2408704,
    671297,
    1046703,
    1053526,
    224396,
    2378928,
    80532,
    77496,
    2879467,
    969132,
    2812797,
    565860,
    994285,
    509362,
    2957038,
    2531131,
    2981837,
    397343,
    2645171,
    2392731,
    801676,
    17900,
    156666,
    1595929,
    2722389,
    2758209,
    107240,
    1635828,
    1630993,
    2436398,
    1250676,
    2864692,
    1606296,
    210359,
    2810255,
    2452117,
    318585,
    465723,
    234559,
    459956,
    650119,
    2964031,
    1634640,
    1387102,
    1117084,
    2813547,
    2680821,
    780624,
    385346,
    2657681,
    2417354,
    693818,
    1041744,
    1055252,
    2019960,
    2310287,
    468866,
    332752,
    2161832,
    135097,
    1212272,
    2226499,
    1044187,
    105309,
    2931981,
    2882533,
    470794,
    2104994,
    665469,
    2787772,
    447192,
    1441922,
    397305,
    2893258,
    1198536,
    1430396,
    218891,
    2320853,
    2952272,
    909056,
    2872478,
    562989,
    914207,
    517677,
    1770231,
    1688830,
    1124530,
    384808,
    851605,
    2294249,
    656141,
    230206,
    2594946,
    2862033,
    1343576,
    292285,
    1437384,
    2970573,
    1343556,
    426318,
    808116,
    493296,
    277247,
    1593590,
    350601,
    2758920,
    286249,
    2011989,
    343733,
    2902595,
    21428,
    1438608,
    2299496,
    1457256,
    2911610,
    2531727,
    643733,
    1373181,
    255772,
    285119,
    1335111,
    1670292,
    461241,
    547956,
    1398888,
    1547018,
    1426138,
    2547033,
    2319760,
    1623705,
    280977,
    2418653,
    2512722,
    207592,
    2832934,
    440152,
    1618367,
    2236458,
    71845,
    566618,
    2418895,
    99967,
    2867128,
    793919,
    2775888,
    2378505,
    1413409,
    2226474,
    525649,
    1456329,
    2874005,
    1788957,
    1625480,
    1245827,
    1937592,
    475154,
    654356,
    2866119,
    27884,
    622551,
    73460,
    956288,
    693,
    224617,
    2667982,
    2534495,
    2909256,
    1406110,
    138143,
    977763,
    2212072,
    17216,
    216059,
    487843,
    996702,
    1038870,
    2850744
]

selected_conversations = amazon_df[
    amazon_df["conversation_id"].isin(
        selected_conversation_ids
    )
].copy()


# ------------------------------------------------------------
# 3. Sort conversations chronologically
# ------------------------------------------------------------

selected_conversations = selected_conversations.sort_values(
    ["conversation_id", "created_at"]
)

In [17]:
def show_conversation(conversation_id):

    conversation = selected_conversations[
        selected_conversations["conversation_id"] == conversation_id
    ]

    print("\n")
    print("=" * 100)
    print(f"CONVERSATION: {conversation_id}")
    print("=" * 100)

    if conversation.empty:
        print("⚠️ Conversation not found.")
        return

    for _, row in conversation.iterrows():

        role = (
            "👤 Customer"
            if row["inbound"]
            else "🏢 AmazonHelp"
        )

        print(f"\n{role}")
        print("-" * 80)
        print(row["clean_text"])

In [ ]:
#Vidhul
show_conversation(863820)
show_conversation(525543)
show_conversation(1231224)
show_conversation(2841686)
show_conversation(2369649)



CONVERSATION: 930181

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Recently moved from Kindle Voyage to Kindle Oasis. Wiped and sold the Voyage, and then realized that my notes weren’t brought over. I thought those were saved to the cloud somewhere. Am I out of luck? Any way to retrieve them from old/wiped device?

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Thanks for contacting us today! We'd like to help if we can! To confirm, are you seeing your notes listed here: but not on the device? ^CS

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp That is correct. I am seeing them on that site. Both the ones on the Voyage, and the couple I’ve done on the Oasis, but on the Oasis, only seeing the ones made on that device.

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER]

In [31]:
import pandas as pd

# ============================================================
# 1. Create complete conversation text
# ============================================================

def build_conversation_text(group):

    messages = []

    for _, row in group.iterrows():

        # Identify speaker
        if row["inbound"] == True:
            role = "Customer"
        else:
            role = "AmazonHelp"

        # Get cleaned message
        text = row["clean_text"]

        messages.append(
            f"{role}: {text}"
        )

    # Join all messages into one conversation
    return "\n\n".join(messages)


# ============================================================
# 2. Build complete text for every conversation
# ============================================================

conversation_texts = (
    amazon_df
    .groupby("conversation_id", sort=False)
    .apply(build_conversation_text)
    .reset_index(name="conversation_text")
)


# ============================================================
# 3. Update the Golden Set
# ============================================================

# Remove the old incomplete conversation_text
golden_set = golden_set.drop(
    columns=["conversation_text"],
    errors="ignore"
)


# Add the complete conversation text
golden_set = golden_set.merge(
    conversation_texts,
    on="conversation_id",
    how="left"
)


# ============================================================
# 4. Arrange columns
# ============================================================

golden_set = golden_set[
    [
        "conversation_id",
        "conversation_text",
        "candidate_intent",
        "candidate_count",
        "group",
        "sampling_type",
        "gold_intent",
        "label_reason",
        "difficulty"
    ]
]


# ============================================================
# 5. Check the result
# ============================================================

print("Golden Set shape:", golden_set.shape)

print(
    "Missing conversation texts:",
    golden_set["conversation_text"].isna().sum()
)

print("\nExample conversation:")
print(golden_set.iloc[0]["conversation_text"])

Golden Set shape: (192, 9)
Missing conversation texts: 0

Example conversation:
Customer: [USER] thanks for the gift card that was unusable. $50 someone wasted 😠

AmazonHelp: [USER] Oh no! We'd like to help if we can Brian! Please contact us via phone or chat here: ^AJ

Customer: @AmazonHelp They escalated and in minutes I had my balance in my account, great service!!!


C:\Users\Nihelesh M U\AppData\Local\Temp\ipykernel_7172\2285066971.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_conversation_text)


In [33]:
# ============================================================
# 6. Save updated Golden Set
# ============================================================

golden_set.to_csv(
    "../data/processed/amazonhelp_golden_set.csv",
    index=False
)

print("✅ amazonhelp_golden_set.csv updated successfully.")

✅ amazonhelp_golden_set.csv updated successfully.


In [35]:
#Mani
show_conversation(1699857)
show_conversation(2283867)
show_conversation(2408704)
show_conversation(671297)
show_conversation(1046703)



CONVERSATION: 1699857

👤 Customer
--------------------------------------------------------------------------------
[USER] No response to my return order for Omega 369 soft gels. expiry too near. Order ID: 406-2352573-1531566 Seller: Health-Mall

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Sorry for the hassle. Please report this to our support team here: and we'll check this. 1/2^HN

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Please don’t provide your order details as it is personal information. Our page is visible to the public.2/2^HN

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp ok


CONVERSATION: 2283867

👤 Customer
--------------------------------------------------------------------------------
[USER] @AmazonHelp utterly disgusting customer service. I ordered my Grandsons main birthday present in plenty time 

In [ ]:
#Sabarish
show_conversation(1053526)
show_conversation(224396)
show_conversation(2378928)
show_conversation(80532)




CONVERSATION: 1053526

👤 Customer
--------------------------------------------------------------------------------
Freaking service by Amazon!!! One of the waste online sites ever seen [USER] @AmazonHelp

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Looks like you had a bad experience with us. Please tell us what went wrong, we will help you. ^BS

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp U people cancel the order by ur choice n then ask us to place a new order wen there is hike in price

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] You should've received one too about this. I've shared your feedback with the concerned team for review. [2/2] ^HA

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] I'm sorry about the order being canceled. Every canceled order is fo

In [40]:
#Siva
show_conversation(77496)
show_conversation(2879467)
show_conversation(969132)
show_conversation(2812797)




CONVERSATION: 77496

👤 Customer
--------------------------------------------------------------------------------
[USER] this is ridiculous...i specified that this order was a gift too....and a "replacement" would cost me more since one of the items was a part of a lightning deal...

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Hi, have you had a chance to request a refund or replacement for your order?^PJ

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp not yet, would a replacement still be of the same amount if one of the items was a part of a lightning deal?

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Great question! Please reach out to us here: so we can look into this together. ^SC

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp ok but this is a link to the

In [ ]:

show_conversation(565860)
show_conversation(994285)
show_conversation(509362)
show_conversation(2957038)

In [38]:
show_conversation(2531131)
show_conversation(2981837)
show_conversation(397343)
show_conversation(2645171)
show_conversation(2392731)
show_conversation(801676)




CONVERSATION: 2531131

👤 Customer
--------------------------------------------------------------------------------
本日も警戒船午前で終了 ウチに帰ると再度購入 アマゾンのFire TV Stickが 届いました。 プライム会員なので見放題 ビリギャル観たけど画像良し👍

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] ご購入、ありがとうございました。プライムビデオのコンテンツをぜひお楽しみください(*'▽') EK


CONVERSATION: 2981837

👤 Customer
--------------------------------------------------------------------------------
I give up. I've just updated my bank details TWICE on amazon and it just keep saying I haven't provided all information and sends me back to the start. Fuck it. I guess I'm not self publishing after all.

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp ama [USER]

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] I'm sorry for the technical difficulties! Please connect with Kindle Direct Publishing here: The availability is 6

In [ ]:
show_conversation(17900)
show_conversation(156666)
show_conversation(1595929)
show_conversation(2722389)
show_conversation(2758209)


In [ ]:
show_conversation(107240)
show_conversation(1635828)
show_conversation(1630993)
show_conversation(2436398)
show_conversation(1250676)

In [ ]:
show_conversation(2864692)
show_conversation(1606296)
show_conversation(210359)
show_conversation(2810255)
show_conversation(2452117)

In [ ]:
show_conversation(318585)
show_conversation(465723)
show_conversation(234559)
show_conversation(459956)
show_conversation(650119)

In [ ]:
show_conversation(2964031)
show_conversation(1634640)
show_conversation(1387102)
show_conversation(1117084)
show_conversation(2813547)


In [ ]:
show_conversation(2680821)
show_conversation(780624)
show_conversation(385346)
show_conversation(2657681)
show_conversation(2417354)

In [ ]:
show_conversation(693818)
show_conversation(1041744)
show_conversation(1055252)
show_conversation(2019960)
show_conversation(2310287)


In [ ]:
show_conversation(468866)
show_conversation(332752)
show_conversation(2161832)
show_conversation(135097)
show_conversation(1212272)


In [ ]:
show_conversation(2226499)
show_conversation(1044187)
show_conversation(105309)
show_conversation(2931981)
show_conversation(2882533)
show_conversation(470794)

In [ ]:
show_conversation(2104994)
show_conversation(665469)
show_conversation(2787772)
show_conversation(447192)
show_conversation(1441922)


In [ ]:
show_conversation(397305)
show_conversation(2893258)
show_conversation(1198536)
show_conversation(1430396)
show_conversation(218891)


In [ ]:
show_conversation(2320853)
show_conversation(2952272)
show_conversation(909056)
show_conversation(2872478)
show_conversation(562989)


In [ ]:
show_conversation(914207)
show_conversation(517677)
show_conversation(1770231)
show_conversation(1688830)
show_conversation(1124530)


In [ ]:
show_conversation(384808)
show_conversation(851605)
show_conversation(2294249)
show_conversation(656141)
show_conversation(230206)


In [ ]:
show_conversation(2594946)
show_conversation(2862033)
show_conversation(1343576)
show_conversation(292285)
show_conversation(1437384)


In [ ]:
show_conversation(2970573)
show_conversation(1343556)
show_conversation(426318)
show_conversation(808116)
show_conversation(493296)


In [ ]:
show_conversation(277247)
show_conversation(1593590)
show_conversation(350601)
show_conversation(2758920)
show_conversation(286249)


In [ ]:
show_conversation(2011989)
show_conversation(343733)
show_conversation(2902595)
show_conversation(21428)
show_conversation(1438608)

In [ ]:
show_conversation(2299496)
show_conversation(1457256)
show_conversation(2911610)
show_conversation(2531727)
show_conversation(643733)

In [ ]:
show_conversation(1373181)
show_conversation(255772)
show_conversation(285119)
show_conversation(1335111)
show_conversation(1670292)
show_conversation(461241)
show_conversation(547956)
show_conversation(1398888)
show_conversation(1547018)
show_conversation(1426138)

In [ ]:
show_conversation(2547033)
show_conversation(2319760)
show_conversation(1623705)
show_conversation(280977)
show_conversation(2418653)



In [25]:
show_conversation(2512722)
show_conversation(207592)
show_conversation(2832934)
show_conversation(440152)
show_conversation(1618367)
show_conversation(2236458)



CONVERSATION: 2512722

👤 Customer
--------------------------------------------------------------------------------
@119625 When will Mr. Robot Season 3 available for streaming?

🏢 AmazonHelp
--------------------------------------------------------------------------------
@716115 We don't have the requested information. Kindly stay tuned for updates.


CONVERSATION: 207592

👤 Customer
--------------------------------------------------------------------------------
@115821 @115830 @AmazonHelp I wish just one of your agent would tell me the same story next time will go to @117249 cheaper and less hassle

🏢 AmazonHelp
--------------------------------------------------------------------------------
@139580 I'm sorry for the disappointment, David! Without including personal/account info, could you tell us more about what's wrong?

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Amazon app says delivered at locker 1st amazon agent say 

In [24]:
show_conversation(71845)
show_conversation(566618)
show_conversation(2418895)
show_conversation(99967)
show_conversation(2867128)




CONVERSATION: 71845

👤 Customer
--------------------------------------------------------------------------------
@115830 been trying to login to my account to place an order, but it won’t let me! What’s going on?! #Irritating

🏢 AmazonHelp
--------------------------------------------------------------------------------
@132121 Hi Zoe, have you checked to make sure your username and password are correct? You could always try to choose the "forgot username/password" option and follow the instructions from there.

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp I’ve already done this and changed my password and it still won’t log me in ‍️

🏢 AmazonHelp
--------------------------------------------------------------------------------
@132121 Sorry to hear this. Please reach out to us here: so we can look into this for you.

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Ok than

In [23]:
show_conversation(793919)
show_conversation(2775888)
show_conversation(2378505)
show_conversation(1413409)
show_conversation(2226474)
show_conversation(525649)



CONVERSATION: 793919

👤 Customer
--------------------------------------------------------------------------------
Hey @115821 all the books I've ever ordered where damaged during delivery. I'm starting to get a bit pissed off about it.

🏢 AmazonHelp
--------------------------------------------------------------------------------
@309299 I'm sorry your items were damaged. You can view your options here:


CONVERSATION: 2775888

👤 Customer
--------------------------------------------------------------------------------
@115830 @116324 Is this your new delivery method? #anywherewilldo #newlaptopthrownovergate

🏢 AmazonHelp
--------------------------------------------------------------------------------
@775646 I'm so sorry for this poor experience, Jenni! Is the laptop okay?

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Seems to be thankfully. Outer box damaged but inner one seems ok. Haven't turned it on yet as it's a christmas

In [21]:
show_conversation(1456329)
show_conversation(2874005)
show_conversation(1788957)
show_conversation(1625480)
show_conversation(1245827)




CONVERSATION: 1456329

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp suspicious email telling me I've signed up to Amazon prime annual membership link to cancel takes me to PayPal account details

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp @AmazonHelp I forwarded to __email__ the email address isn't but at first glance looks like it is

🏢 AmazonHelp
--------------------------------------------------------------------------------
@458022 Thank you so much for taking the time to do that, Charlotte!


CONVERSATION: 2874005

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Phishing email from __email__

🏢 AmazonHelp
--------------------------------------------------------------------------------
@797896 Danke für die Info. Bitte gleich löschen! Liebe Grüße und einen schönen Tag

👤 Customer
---------------------------

In [20]:
show_conversation(1937592)
show_conversation(475154)
show_conversation(654356)
show_conversation(2866119)
show_conversation(27884)
show_conversation(622551)




CONVERSATION: 1937592

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp My package was supposed to be here on the 9th! The app is telling me that y'all tried to give it to me but I didn't respond?

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp I've been home the last two days. No one has attempted anything. Not even a note on my door-nothing. Where is my package???

🏢 AmazonHelp
--------------------------------------------------------------------------------
@576395 Oh no! That's not what we want to hear. So we can relay the correct information, could you tell us who the carrier for this delivery was? You can find it here:

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp I got it today! But y'all charged me twice for it?? Once to me, my default card, and once to my roommate. Not my default. I only ordered one

🏢 Amazo

In [19]:
show_conversation(73460)
show_conversation(956288)
show_conversation(693)
show_conversation(224617)
show_conversation(2667982)
show_conversation(2534495)




CONVERSATION: 73460

👤 Customer
--------------------------------------------------------------------------------
This is what i received, broken car steering mobile holder from @115821 @AmazonHelp @115850 Order id : 402-4901639-7214711. This was unexpected from such a big brand

🏢 AmazonHelp
--------------------------------------------------------------------------------
@132428 Sorry to know you've received a faulty product, Prem. You can create a return request here: Also, please don't provide your order details, we consider it to be personal information. Our page is visible to the public.

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp This is what it saying. I just want to replace it.

🏢 AmazonHelp
--------------------------------------------------------------------------------
@132428 As per the screenshot, the item in picture isn't eligible for return. You can refer to our return policies here : Appreciate your understand

In [18]:
show_conversation(2909256)
show_conversation(1406110)
show_conversation(138143)
show_conversation(977763)
show_conversation(2212072)




CONVERSATION: 2909256

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp My account was hacked. Someone changed the email address associated w/ my account & I cannot access my account. I am a Prime member, have both Prime credit cards & use streaming services. I’ve called customer service 3 times since 11/17. Still can’t access account.

🏢 AmazonHelp
--------------------------------------------------------------------------------
@805454 I'm so sorry for the ongoing trouble with your account, Travis! I'd like to have a member of our team look into this on your behalf. Please "skip sign in" and send us the details here:

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Thank you thank you thank you thank you!! I just submitted the info and I really appreciate your help! I just want to log into my account, use prime services, stream video, etc.!!!

🏢 AmazonHelp
-----------------

In [39]:
show_conversation(17216)
show_conversation(216059)
show_conversation(487843)
show_conversation(996702)
show_conversation(1038870)
show_conversation(2850744)




CONVERSATION: 17216

👤 Customer
--------------------------------------------------------------------------------
[USER] what is the use of Prime Membership if I am not getting my parcel on time.

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] We're sorry for the delay in the delivery of your order. Please contact us here: and we'll help you ^ZH

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp Pathetic reply by your caller

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Did we happen to miss the estimated delivery date? ^AG

👤 Customer
--------------------------------------------------------------------------------
@AmazonHelp No use of prime membership, return my money

🏢 AmazonHelp
--------------------------------------------------------------------------------
[USER] Could you please confirm, if we've missed the estimate